# B2-019 — Practice p11

**Set:** B · **Type:** constrained-coding · **Difficulty:** advanced · **Minutes:** 50

**Concepts:** scaled-dot-product-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Task

Implement class `ScaledMaskedAttention(torch.nn.Module)` with no trainable parameters. Its `forward(q,k,v,allowed)` returns `(weights, output)`. Inputs have shapes `(B,Nq,Dk)`, `(B,Nk,Dk)`, `(B,Nk,Dv)`; Boolean `allowed` must broadcast to `(B,Nq,Nk)`. Scale by `sqrt(Dk)`, apply the mask before softmax, and raise `ValueError` if any broadcast mask row is all false. Do not mutate any supplied tensor.

Run the fixed probe below and compare both return values with `expected_weights` and `expected_output`. Then run `reject_allowed`, whose second query row is all false, and certify rejection. Return variables named `weights` and `output`.

Pinned contract: seed `20260808`; CPU dtype `float64`; exact shapes `q=(1,2,2)`, `k=(1,3,2)`, `v=(1,3,2)`, `allowed=(1,2,3)`, `weights=(1,2,3)`, and `output=(1,2,2)`; `atol=1e-10`, `rtol=1e-10`. Allowed APIs: `torch.matmul`, `transpose`, `masked_fill`, `softmax`, validation. Banned APIs: `torch.nn.functional.scaled_dot_product_attention`, `torch.nn.MultiheadAttention`, CUDA/MPS, detach-to-NumPy inside forward.

In [ ]:
import math

import numpy as np
import torch

torch.manual_seed(20260808)
q = torch.tensor([[[1.0, 2.0], [-1.0, 0.5]]], dtype=torch.float64)
k = torch.tensor(
    [[[2.0, 0.0], [0.0, 1.0], [1.0, -1.0]]], dtype=torch.float64
)
v = torch.tensor(
    [[[2.0, -1.0], [0.0, 3.0], [4.0, 1.0]]], dtype=torch.float64
)
allowed = torch.tensor(
    [[[True, True, False], [True, False, True]]], dtype=torch.bool
)
reject_allowed = torch.tensor(
    [[[True, True, False], [False, False, False]]], dtype=torch.bool
)

q_np, k_np, v_np = q.numpy(), k.numpy(), v.numpy()
allowed_np = allowed.numpy()
scores_np = q_np @ np.swapaxes(k_np, -1, -2) / math.sqrt(q.shape[-1])
masked_np = np.where(allowed_np, scores_np, -np.inf)
shifted_np = masked_np - np.max(masked_np, axis=-1, keepdims=True)
exp_np = np.exp(shifted_np)
expected_weights = exp_np / np.sum(exp_np, axis=-1, keepdims=True)
expected_output = expected_weights @ v_np
assert expected_weights.shape == (1, 2, 3)
assert expected_output.shape == (1, 2, 2)

In [ ]:
# Write your implementation and run both required probes here.


## Your response

Show the requested derivation, implementation, audit, or justification here.